# Lab 7 — Large Language Models (LLMs)

**Student name:** Faisal AL Zahrani  
**Student ID:** 2230000363

**Objective:** Compare decoder-only, encoder–decoder, and encoder-only Transformers through text generation, translation, sentiment classification, and summarization.

All three worked examples, Tasks **1, 2, 3, and 5**, and the **optional challenge** are completed in the source order. The source notebook contains no Task 4; its numbering is preserved. Model outputs are saved from an actual CPU run.

# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


**Setup.** Use Python 3.12 and the accompanying `requirements.txt`. This replaces the source's unpinned shell installation with an explicit dependency check. The three public model revisions are fixed for reproducibility. Their files are saved in `models/`; they download on first use when absent. Subsequent complete local copies run offline, without an API key or paid inference service.

In [1]:
import importlib.util
import importlib.metadata
from pathlib import Path
import json
import os
import sys

# Keep model downloads inside the lab directory.
os.environ['HF_HOME'] = str(Path('.cache/huggingface').resolve())
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'

required = ['torch', 'transformers', 'sentencepiece', 'huggingface_hub', 'safetensors', 'pandas', 'ipywidgets']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Install requirements.txt in Python 3.12 and select that kernel. Missing: ' + ', '.join(missing))

import pandas as pd
import torch
from IPython.display import display, Markdown
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
pd.set_option('display.max_colwidth', 180)
MODEL_ROOT = Path('models')
MODEL_SPECS = {'decoder': {'repo_id': 'distilbert/distilgpt2', 'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa', 'folder': 'distilgpt2'}, 'seq2seq': {'repo_id': 'google/flan-t5-small', 'revision': '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab', 'folder': 'flan-t5-small'}, 'encoder': {'repo_id': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english', 'revision': '714eb0fa89d2f80546fda750413ed43d93601a13', 'folder': 'distilbert-base-uncased-finetuned-sst-2-english'}}
MODEL_FILES = ['README.md', 'LICENSE', 'LICENSE.txt', 'config.json', 'generation_config.json',
               'model.safetensors', 'tokenizer.json', 'tokenizer_config.json',
               'special_tokens_map.json', 'spiece.model', 'merges.txt', 'vocab.json', 'vocab.txt']

def ensure_model(key):
    spec = MODEL_SPECS[key]
    local = MODEL_ROOT / spec['folder']
    marker = local / 'lab_revision.json'
    if marker.is_file():
        saved = json.loads(marker.read_text(encoding='utf-8'))
        files = saved.get('files', {})
        if (saved.get('revision') == spec['revision'] and files
                and {'config.json', 'model.safetensors'}.issubset(files)
                and all((local / name).is_file() and (local / name).stat().st_size == item['bytes']
                        for name, item in files.items())):
            print('Using local model:', spec['repo_id'])
            return str(local)
    print('Downloading model:', spec['repo_id'])
    snapshot_download(repo_id=spec['repo_id'], revision=spec['revision'],
                      local_dir=local, allow_patterns=MODEL_FILES, token=False)
    files = {name: {'bytes': (local / name).stat().st_size}
             for name in MODEL_FILES if (local / name).is_file()}
    assert {'config.json', 'model.safetensors'}.issubset(files)
    marker.write_text(json.dumps({**spec, 'files': files}, indent=2), encoding='utf-8')
    return str(local)

print('Python:', sys.version.split()[0])
for package in ['torch', 'transformers', 'sentencepiece', 'huggingface-hub', 'safetensors', 'pandas']:
    print(f'{package}: {importlib.metadata.version(package)}')
print('Device: CPU; all inference runs locally.')

Python: 3.12.14
torch: 2.14.1+cpu
transformers: 4.57.6
sentencepiece: 0.2.2
huggingface-hub: 0.36.2
safetensors: 0.8.0
pandas: 3.0.1
Device: CPU; all inference runs locally.


In [2]:
from transformers import (
    pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
)
print('Libraries loaded successfully!')

Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
decoder_path = ensure_model('decoder')
gpt_tokenizer = AutoTokenizer.from_pretrained(decoder_path, local_files_only=True)
gpt_model = AutoModelForCausalLM.from_pretrained(
    decoder_path, local_files_only=True, use_safetensors=True, trust_remote_code=False
)
gpt_model.eval()
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
generator = pipeline('text-generation', model=gpt_model, tokenizer=gpt_tokenizer,
                     framework='pt', device=-1)
prompt = 'Natural language processing is'
output = generator(prompt, max_new_tokens=30, do_sample=False, num_beams=1, temperature=1.0,
                   pad_token_id=gpt_tokenizer.eos_token_id)
greedy_text = output[0]['generated_text']
print(greedy_text.rstrip())  # Trim trailing blank lines for display only.
assert greedy_text.startswith(prompt) and len(greedy_text) > len(prompt)

Device set to use cpu


Using local model: distilbert/distilgpt2


Natural language processing is a common feature of the language processing process.


**Observation.** With `do_sample=False` and one beam, each step chooses the highest-scoring next token. The result can be repetitive: DistilGPT-2 is a small continuation model, not an instruction-tuned assistant, and fluent output is not evidence of factual accuracy.

# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
seq2seq_path = ensure_model('seq2seq')
tokenizer = AutoTokenizer.from_pretrained(seq2seq_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    seq2seq_path, local_files_only=True, use_safetensors=True, trust_remote_code=False
)
model.eval()
translation_prompt = 'Translate to French: I love natural language processing.'
translation_inputs = tokenizer(translation_prompt, return_tensors='pt')
with torch.inference_mode():
    translation_ids = model.generate(**translation_inputs, max_new_tokens=40,
                                     do_sample=False, num_beams=1)
translation = tokenizer.decode(translation_ids[0], skip_special_tokens=True)
print('Prompt:', translation_prompt)
print('French:', translation)
assert translation.strip()

Using local model: google/flan-t5-small


Prompt: Translate to French: I love natural language processing.
French: Je s'agissant de la technologie de linguage naturelle.


**Translation review.** The saved FLAN-T5 output is grammatically unreliable and does not faithfully express “I love.” A human-written reference translation is **“J'aime le traitement automatique du langage naturel.”** This reference is provided for comparison and is not presented as model output. Successfully executing a translation model does not guarantee a correct translation.

# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
encoder_path = ensure_model('encoder')
sentiment_tokenizer = AutoTokenizer.from_pretrained(encoder_path, local_files_only=True)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    encoder_path, local_files_only=True, use_safetensors=True, trust_remote_code=False
)
sentiment_model.eval()
classifier = pipeline('sentiment-analysis', model=sentiment_model, tokenizer=sentiment_tokenizer,
                      framework='pt', device=-1)
text = 'This NLP lab is easy and interesting.'
result = classifier(text)
print('Text: ', text)
print('Label:', result[0]['label'])
print(f"Score: {result[0]['score']:.6f}")

Device set to use cpu


Using local model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Text:  This NLP lab is easy and interesting.
Label: POSITIVE
Score: 0.999779


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

**Task 1 — Answer.** The table gives the conventional best fit among the three families taught in this lab. These are common choices, not exclusive capabilities: appropriately trained models from other families can also perform several of these tasks.

In [6]:
family_answers = [
    ('Sentiment classification', 'Encoder-only', 'DistilBERT',
     'Bidirectional context supports a label for the full input.'),
    ('Machine translation', 'Encoder-decoder', 'T5 / BART',
     'Encode the source sentence and decode a target-language sequence.'),
    ('Open-ended text generation', 'Decoder-only', 'GPT / DistilGPT-2',
     'Generate a continuation by predicting the next token repeatedly.'),
    ('Named Entity Recognition', 'Encoder-only', 'BERT',
     'Contextual token representations support an entity label for each token.'),
    ('Text summarization', 'Encoder-decoder', 'T5 / BART',
     'Encode the source document and generate a shorter output sequence.'),
]
family_table = pd.DataFrame(family_answers, columns=['Task', 'Model family', 'Example', 'Reason'])
display(family_table)

,Task,Model family,Example,Reason
0,Sentiment classification,Encoder-only,DistilBERT,Bidirectional context supports a label for the full input.
1,Machine translation,Encoder-decoder,T5 / BART,Encode the source sentence and decode a target-language sequence.
2,Open-ended text generation,Decoder-only,GPT / DistilGPT-2,Generate a continuation by predicting the next token repeatedly.
3,Named Entity Recognition,Encoder-only,BERT,Contextual token representations support an entity label for each token.
4,Text summarization,Encoder-decoder,T5 / BART,Encode the source document and generate a shorter output sequence.


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

**Task 2 — Method.** Apply the same DistilBERT classifier to all three specified sentences and print the label and score. The score is the model's softmax value for the selected class; it is not a guarantee of correctness or a measured accuracy.

In [7]:
task2_sentences = [
    'The movie was excellent.',
    'The lecture was difficult to understand.',
    'I enjoyed learning about transformers.',
]
task2_predictions = classifier(task2_sentences, batch_size=3)
for sentence, prediction in zip(task2_sentences, task2_predictions):
    print(f"Sentence: {sentence}\nLabel: {prediction['label']} | Score: {prediction['score']:.6f}\n")
assert len(task2_predictions) == 3
assert all(p['label'] in {'POSITIVE', 'NEGATIVE'} and 0 <= p['score'] <= 1
           for p in task2_predictions)

Sentence: The movie was excellent.
Label: POSITIVE | Score: 0.999861

Sentence: The lecture was difficult to understand.
Label: NEGATIVE | Score: 0.999665

Sentence: I enjoyed learning about transformers.
Label: POSITIVE | Score: 0.999746



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


**Task 3 — Method.** Provide a short original paragraph and an explicit summarization instruction to FLAN-T5-small. Four-beam decoding with sampling disabled produces a repeatable candidate summary in the tested environment. The generated summary is shown exactly as returned by the model.

In [8]:
# Original paragraph written for this exercise; no external dataset is needed.
paragraph = (
    'The university library opened a new digital learning center this week. '
    'The center has computers, quiet study rooms, and free workshops on programming and data analysis. '
    'Students can reserve equipment online and ask librarians for technical support. '
    'The project aims to make technology more accessible and support independent study.'
)
summary_prompt = 'Summarize the following paragraph in one sentence: ' + paragraph
summary_inputs = tokenizer(summary_prompt, return_tensors='pt')
assert summary_inputs['input_ids'].shape[1] <= tokenizer.model_max_length
with torch.inference_mode():
    summary_ids = model.generate(**summary_inputs, max_new_tokens=60, do_sample=False,
                                 num_beams=4, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print('Original paragraph:')
print(paragraph)
print('\nFLAN-T5 summary:')
print(summary)
print(f'\nWord counts: original={len(paragraph.split())}, summary={len(summary.split())}')
assert summary.strip()
assert len(summary.split()) < len(paragraph.split())

Original paragraph:
The university library opened a new digital learning center this week. The center has computers, quiet study rooms, and free workshops on programming and data analysis. Students can reserve equipment online and ask librarians for technical support. The project aims to make technology more accessible and support independent study.

FLAN-T5 summary:
The university library has opened a new digital learning center.

Word counts: original=49, summary=10


**Summary review.** The saved summary captures the central event—the library opening a digital learning center—and reduces 49 words to 10. It omits supporting details about equipment, workshops, and accessibility. This is a qualitative check on one example, not a general summarization-quality score.


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer (4 sentences):** A decoder-only model generates a response one token at a time, conditioning each prediction on the conversation supplied in its context. Causal self-attention lets it use earlier turns while generating a coherent continuation. Pretraining develops broad language patterns, and instruction tuning plus human-feedback training can adapt this ability to follow user requests and conversational conventions. This makes the architecture suitable for chatbots such as ChatGPT, although the architecture alone does not guarantee factual or reliable responses.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


**Optional challenge — Experiment.** Run the same prompt three times with sampling enabled. The following cell deliberately contains no `set_seed(42)` call and does not reset the random-number generator between requests. Running this cell again advances the generator further and can produce new continuations.

In [9]:
sampling_options = dict(
    max_new_tokens=30, do_sample=True, num_beams=1,
    temperature=0.8, top_k=50, top_p=0.92,
    pad_token_id=gpt_tokenizer.eos_token_id,
)
sampled_outputs = []
for run in range(1, 4):
    generated = generator(prompt, **sampling_options)[0]['generated_text']
    sampled_outputs.append(generated)
    print(f'Run {run}: {generated.rstrip()}\n')
unique_samples = len(set(sampled_outputs))
print(f'Distinct outputs observed: {unique_samples}/3')
assert len(sampled_outputs) == 3
assert all(text.startswith(prompt) for text in sampled_outputs)

Run 1: Natural language processing is often used to address the need to do the processing yourself. This can be done by simply opening a terminal in a terminal window and typing in a word



Run 2: Natural language processing is not required. If you are not familiar with the language processing, please refer to the following information about the processing process for more information.



Run 3: Natural language processing is a major challenge for many languages. If you have ever been a computer programmer or programmer, you may want to consider some other languages.

Distinct outputs observed: 3/3


**Answer.** Sampling draws the next token from a probability distribution instead of always taking the most likely token. Different draws change the continuation, and each newly chosen token changes the context for later predictions. Temperature, top-k, and top-p control the distribution used for these draws. Without resetting the seed, repeated calls can differ, although identical outputs are still possible by chance. Resetting the same seed before each call can reproduce a sampled result when the model, settings, and execution environment are unchanged.

**Controlled comparison.** Reset the seed before each of two additional sampling calls, then repeat the greedy example. This separates randomness introduced by sampling from a change in the prompt or model.

In [10]:
set_seed(42)
seeded_first = generator(prompt, **sampling_options)[0]['generated_text']
set_seed(42)
seeded_second = generator(prompt, **sampling_options)[0]['generated_text']
greedy_repeat = generator(prompt, max_new_tokens=30, do_sample=False, num_beams=1, temperature=1.0,
                          pad_token_id=gpt_tokenizer.eos_token_id)[0]['generated_text']
print('Seeded sample 1:', seeded_first)
print('Seeded sample 2:', seeded_second)
print('Same seed, same sampled output:', seeded_first == seeded_second)
print('Repeated greedy output matches:', greedy_repeat == greedy_text)
assert seeded_first == seeded_second
assert greedy_repeat == greedy_text

Seeded sample 1: Natural language processing is a technique used to create new semantic processing tools that can be applied to the language and to create new semantic processing tools that can be applied to the language
Seeded sample 2: Natural language processing is a technique used to create new semantic processing tools that can be applied to the language and to create new semantic processing tools that can be applied to the language
Same seed, same sampled output: True
Repeated greedy output matches: True


### Completion checklist

| Requirement | Completed work |
|---|---|
| Part A | DistilGPT-2 continuation of the exact source prompt with 30 new tokens and no sampling. |
| Part B | FLAN-T5-small translation of the original English sentence to French. |
| Part C | DistilBERT sentiment prediction for the original example. |
| Task 1 | All five task-to-family choices, with examples and reasons. |
| Task 2 | Label and score for each of the three exact sentences. |
| Task 3 | Original short paragraph, FLAN-T5 summary, and word counts. |
| Task 5 | Four-sentence reflection about decoder-only chatbots. |
| Optional challenge | Three sampled continuations, an explanation of variation, and seeded/greedy controls. |

### References

- [DistilGPT-2 model card](https://huggingface.co/distilbert/distilgpt2)
- [FLAN-T5-small model card](https://huggingface.co/google/flan-t5-small)
- [DistilBERT SST-2 sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)
- [Transformers generation options](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation)

Model outputs demonstrate behavior on these inputs; this lab does not estimate general task accuracy or summary quality with a benchmark.